# Pipeline 

In [ ]:
%load_ext autoreload
%autoreload 2

from averitec import Datapoint
from evidence_generation import GptEvidenceGenerator, GptBatchedEvidenceGenerator, DynamicFewShotBatchedEvidenceGenerator
from classification import DefaultClassifier, HuggingfaceClassifier, AverageEnsembleClassifier, LogRegEnsembleClassifier
from retrieval import SimpleFaissRetriever, Retriever, MmrFaissRetriever, SubqueryRetriever
from pipeline import Pipeline, MockPipeline
import pickle
from labels import label2id, id2label
import numpy as np
from sklearn.metrics import classification_report
import random
from tqdm import tqdm
import htmldate
random.seed(111)

import json

## Pipeline test

In [ ]:
split = "test"
path = "/mnt/data/factcheck/averimatec/"
with open(path + f"{split}.json") as f:
    dataset = json.load(f)
    for i in range(len(dataset)):
        dataset[i]["claim_id"] = i
        dataset[i]["split"] = split
    datapoints = [Datapoint.from_dict(d) for d in dataset]

In [ ]:
datapoint = Datapoint.from_dict(dataset[150])
datapoint

In [ ]:
print(datapoint.speaker)

In [ ]:
# retriever = SimpleFaissRetriever(path="/mnt/data/factcheck/averitec-data/data_store/vecstore/dev/6k")
retriever = MmrFaissRetriever(path=f"/mnt/data/factcheck/averimatec/vector_store/{split}/text/2k_mxbai",k=7, fetch_k=24)
# retriever = MmrFaissRetriever(path=f"/mnt/data/factcheck/averitec-data/data_store/vecstore/{split}/2k")
retrieval_result = retriever(datapoint)
retrieval_result

In [ ]:
retrieval_result.images

In [ ]:
from classification import DefaultClassifier, HuggingfaceClassifier, AverageEnsembleClassifier, LogRegEnsembleClassifier, RandomForestClassifier, NoTiebreakClassifier

# target = path + "data_store/vecstore/test/2k"
PIPELINE_NAME = "mmr+gpt51-dfewshot-atype"
classifier = NoTiebreakClassifier()  # DefaultClassifier()
if False:
    pipeline = MockPipeline(
        dumps=f"/mnt/data/factcheck/averitec-data/data_store/submissions/{split}_mmr+gpt4o-dfewshot-tiebrk-atype.pkl",
        classifier=NoTiebreakClassifier()
    )
else:
    pipeline = Pipeline(
        #dumps = "/mnt/data/factcheck/averitec-data/data_store/submissions/dev_mmr+gpt4o-dfewshot.pkl",
        #SubqueryRetriever(retriever),
        retriever,
        evidence_generator=DynamicFewShotBatchedEvidenceGenerator(), 
        classifier=classifier
    )

In [ ]:
list(enumerate(retrieval_result.images))

In [ ]:
PIPELINE_NAME = "base64"

In [ ]:
split

In [ ]:
submission = []
dump = []

for dp in tqdm(datapoints):
    pipeline_result = pipeline(dp)
    submission.append(pipeline_result.to_submission())
    dump.append(pipeline_result)
with open(f"{path}submissions/{split}_{PIPELINE_NAME}_3.json", "w") as f:
    json.dump(submission, f, indent=4)
with open(f"{path}submissions/{split}_{PIPELINE_NAME}_3.pkl", "wb") as f:
    pickle.dump(dump, f)

In [ ]:
files = pipeline.evidence_generator.get_batch_files(path=f"/mnt/data/factcheck/averimatec/batch_jobs/{split}_{PIPELINE_NAME}", batch_size=352)

In [ ]:
files

In [ ]:
batch_results = pipeline.evidence_generator.submit_and_await_batches(
    files, f"/mnt/data/factcheck/averimatec/batch_jobs/{split}_{PIPELINE_NAME}/output.jsonl"
)

In [ ]:
with open(f"/mnt/data/factcheck/averimatec/batch_jobs/{split}_{PIPELINE_NAME}/output.jsonl") as f:
    batch_results = [json.loads(line)["response"]["body"]["choices"][0]["message"]["content"] for line in f]

In [ ]:
new_dump = []
pipeline.evidence_generator.fallback_gpt_generator.client.temperature = 0.5
for pipeline_result, batch_result in zip(dump[: len(batch_results)], batch_results):

    new_result = pipeline.evidence_generator.update_pipeline_result(
        pipeline_result, batch_result, pipeline.classifier
    )
    new_dump.append(new_result)

In [ ]:
json.loads("""{
  "questions": [
    {
      "question": "Does [IMG] show the man’s index finger without election ink?",
      "answer": "Yes",
      "source": "11",
      "answer_type": "Boolean",
      "evidence_text": "In [IMG], the man in the foreground points his index finger toward the camera and it does not appear to bear any visible election ink."
    },
    {
      "question": "Does any source explicitly identify the man in [IMG] as ex-opposition supporter Ny Nak?",
      "answer": "No explicit identification in the provided sources",
      "source": "11",
      "answer_type": "Unanswerable",
      "evidence_text": "The available sources and [IMG] do not provide text or captions that explicitly name the man as ex-opposition supporter Ny Nak."
    },
    {
      "question": "Do the provided sources state that the photo in [IMG] was taken during the 2023 Cambodia national election?",
      "answer": "No, the date or specific election year of the photo is not given",
      "source": "11",
      "answer_type": "Unanswerable",
      "evidence_text": "None of the sources linked to [IMG] specifies that it was taken during the 2023 Cambodia national election."
    },
    {
      "question": "Do the textual sources describe low voter turnout in the 2023 Cambodia national election?",
      "answer": "No, they mainly describe restrictions and pressure around voting but do not characterize 2023 turnout as low",
      "source": "4",
      "answer_type": "Abstractive",
      "evidence_text": "Source 4 notes that about 9.7 million Cambodians were registered to vote in 2023 and discusses repression and pressure to vote, but does not describe nationwide turnout as low."
    },
    {
      "question": "Do the textual sources provide overall turnout figures for the 2018 Cambodian election, and if so, were they high?",
      "answer": "Yes, turnout in 2018 is reported at more than 82 percent of registered voters",
      "source": "6",
      "answer_type": "Extractive",
      "evidence_text": "Source 6 reports that more than 82 percent of those registered to vote cast a ballot in the 2018 election, indicating high turnout."
    },
    {
      "question": "Do the sources report that some Cambodians feared being seen without inked fingers after voting days in recent elections?",
      "answer": "Yes",
      "source": "6",
      "answer_type": "Boolean",
      "evidence_text": "Source 6 quotes voters saying they were afraid local authorities would spot them if they didn’t have inked fingers, showing that inked fingers were monitored."
    },
    {
      "question": "Do any sources state that Ny Nak did not vote in the 2023 Cambodia national election?",
      "answer": "No such statement is present in the provided sources",
      "source": "11",
      "answer_type": "Unanswerable",
      "evidence_text": "Among the provided sources, none mentions Ny Nak’s individual voting behavior in the 2023 election."
    },
    {
      "question": "Do any sources explain why the person in [IMG] did or did not vote in the depicted election?",
      "answer": "No, the motives or actions of the person in the image are not described",
      "source": "11",
      "answer_type": "Unanswerable",
      "evidence_text": "The sources with [IMG] do not contain accompanying text that states whether the person voted or his reasons related to turnout."
    },
    {
      "question": "Do the sources indicate that calls for boycotts or clean-finger campaigns existed in Cambodian elections prior to 2023?",
      "answer": "Yes, there were boycott campaigns such as the 2018 “clean finger” campaign urging people not to vote",
      "source": "2",
      "answer_type": "Abstractive",
      "evidence_text": "Source 2 notes that around the 2018 elections, social media campaigns like the “clean finger campaign” called for people to boycott the polls."
    },
    {
      "question": "Do the sources indicate that Cambodian authorities use indelible ink on voters’ index fingers in national elections?",
      "answer": "Yes, indelible ink is used to mark voters’ fingers after they cast ballots",
      "source": "6",
      "answer_type": "Extractive",
      "evidence_text": "Source 6 explains that voters feared being spotted by local authorities if they didn’t have an inked finger, referring to the ink used to mark fingers after voting."
    }
  ],
  "claim_veracity": {
    "Supported": "1",
    "Refuted": "2",
    "Not Enough Evidence": "5",
    "Conflicting Evidence/Cherrypicking": "1"
  },
  "veracity_verdict": "Not Enough Evidence",
  "verdict_justification": "The image does show an uninked index finger in front of a Cambodian polling station, and sources confirm that finger-inking is standard in Cambodian elections. However, none of the provided sources identify the person as ex-opposition supporter Ny Nak, tie this specific photo to the 2023 national election, or state that he did not vote because of low turnout. Turnout data for 2023 are not clearly characterized as low in these sources. Therefore the composite claim cannot be verified or refuted with the available evidence."
}""")

In [ ]:
with open(f"{path}submissions/{split}_{PIPELINE_NAME}2.json", "w") as f:
    json.dump([d.to_submission() for d in new_dump], f, indent=4)
with open(f"{path}submissions/{split}_{PIPELINE_NAME}2.pkl", "wb") as f:
    pickle.dump(new_dump, f)

In [ ]:
f"{path}submissions/{split}_{PIPELINE_NAME}.json"

In [ ]:
(
    new_dump[1],
    new_dump[1].evidence_generation_result,
    new_dump[1].classification_result
)

In [ ]:
with open(f"/mnt/data/factcheck/averitec-data/data_store/submissions/{split}_{PIPELINE_NAME}.json", "w") as f:
    json.dump([d.to_submission() for d in new_dump], f, indent=4)
with open(f"/mnt/data/factcheck/averitec-data/data_store/submissions/{split}_{PIPELINE_NAME}.pkl", "wb") as f:
    pickle.dump(new_dump, f)

In [ ]:
with open(f"/mnt/data/factcheck/averitec-data/data_store/submissions/{split}_{PIPELINE_NAME}.json", "w") as f:
    json.dump([d.to_submission() for d in new_dump], f, indent=4)
with open(f"/mnt/data/factcheck/averitec-data/data_store/submissions/{split}_{PIPELINE_NAME}.pkl", "wb") as f:
    pickle.dump(new_dump, f)

## collapsible begin

In [ ]:
from IPython.display import display, Markdown, Latex

In [ ]:
knn_retrieval_result = retriever(datapoint)
display(Markdown("### 🗯️ " + datapoint.claim))
display(Markdown("*Retrieved by knn*\n\n"))
# sample 3
for r in knn_retrieval_result:
    newline = "\n"
    display(Markdown(f"**{r.metadata['url']}**\n\n{r.page_content[:256]}"))

In [ ]:
from retrieval import MmrFaissRetriever

mmr_retriever = MmrFaissRetriever(retriever.path)
mmr_retrieval_result = mmr_retriever(datapoint)
display(Markdown("### 🗯️ " + datapoint.claim))
display(Markdown("*Retrieved by MMR*\n\n"))
# sample 3
for r in mmr_retrieval_result:
    newline = "\n"
    display(Markdown(f"**{r.metadata['url']}**\n\n{r.page_content[:256]}"))

In [ ]:
subquery_retriever = SubqueryRetriever(retriever)
subquery_retrieval_result = subquery_retriever(datapoint)
display(Markdown("### 🗯️ " + datapoint.claim))
display(Markdown("*Retrieved by subqueries*\n\n"))
# sample 3
for r in subquery_retrieval_result:
    newline = "\n"
    display(Markdown(f"**{r.metadata['url']}**\n\n*{';'.join(r.metadata['queries'])}*\n\n{r.page_content[:256]}"))

In [ ]:
subquery_retrieval_result.metadata

## Collapsible section end

In [ ]:
evidence_generator = GptBatchedEvidenceGenerator("gpt-4o")
evidence_generation_result = evidence_generator(datapoint, retrieval_result)
evidence_generation_result

In [ ]:
evidence_generation_result.metadata["suggested_label"]

In [ ]:
datapoint.label

In [ ]:
classifier = DefaultClassifier()
classification_result = classifier(datapoint, evidence_generation_result, retrieval_result)
str(classification_result), classification_result

In [ ]:
datapoint2 = Datapoint.from_dict(dataset[16])
pipeline = Pipeline(retriever, evidence_generator, classifier)
pipeline_result = pipeline(datapoint2)
pipeline_result

In [ ]:
str(pipeline_result.classification_result), datapoint2.label

In [ ]:
pipeline_result.to_submission()

In [ ]:
# pickle dump pipeline result
import pickle
with open('data/pipeline_result.pkl', 'wb') as f:
    pickle.dump(pipeline_result, f)

In [ ]:
%run src/prediction/evaluate_veracity.py --label_file /mnt/data/factcheck/averitec-data/data/dev.json --prediction_file /mnt/data/factcheck/averitec-data/data_store/submission_dev_avg_clf.json

In [ ]:
import json, os
# crawl /mnt/data/factcheck/averitec-data/data_store/batch_jobs and each time you find gpt4o in folder name and "output" in filename, load the file and add its line count to line_counts
line_counts = []
for root, dirs, files in os.walk("/mnt/data/factcheck/averitec-data/data_store/batch_jobs"):
    #print(root, files)
    if "gpt4o" in root:
        for f in files:
            if "output" in f:
                print(os.path.join(root,f))
                with open(os.path.join(root,f)) as f:
                    line_counts.append(len(f.readlines()))
            

In [ ]:
line_counts

In [ ]:
()/sum(line_counts)